# Building a Handbags-Shoes Classifier with Transfer Learning (PyTorch)

This is the PyTorch version of the Keras [course notebook](course_notebook_transfer_learning.ipynb).

* Based on [Deep Learning with Python](https://www.manning.com/books/deep-learning-with-python-second-edition?gclid=CjwKCAjw9aiIBhA1EiwAJ_GTSlKgxc4qopKHPsFWryOoTz7fvhvhzYSjEsgQ-bG1R51QSGppISywpBoClcIQAvD_BwE) by Francois Chollet

## Introduction

In this notebook, we will first build a **convolutional neural network** from scratch.

We will then describe a very powerful technique called **Transfer Learning** that can be used to build highly accurate image classification models even when you have very little data. Pretty much any consumer-facing app that uses image AI was probably built using this technique.

We also show how to use a technique called **data augmentation** to effortlessly increase the size of your training dataset, and thereby achieve better accuracy.

---


But, as usual, let's get the usual technical preliminaries out of the way first.

In [ ]:
import os
import json
import math
import shutil
import pathlib
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader

import torchvision
from torchvision import transforms, datasets
from torchvision.models import (
    resnet50, ResNet50_Weights,
    vit_b_16, ViT_B_16_Weights,
)

from PIL import Image

torch.manual_seed(42)
np.random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Unlike the earlier notebook where we used a dataset that comes packaged with Keras, in this notebook we will work with our own data.

I have web-scraped approximately 100 color images each of handbags and shoes, packaged it up as a zip file and placed it in Dropbox.

The code below downloads this zip file and unzips it so that we can access it.

In [ ]:
import os
if not os.path.exists("data/handbags-shoes"):
    !wget -q -P ./ https://www.dropbox.com/s/w07liww46kgxo1m/handbags-shoes.zip
    !unzip -qq handbags-shoes.zip -d data/

## Data Pre-processing


Is "Deep Learning" even possible with so few examples? (for comparison: Fashion MNIST had **60,000** training examples).

YES!

We will now build a near-perfect handbags vs shoes classifier using **just** these 200 images!




---



---

Since this isn't a standard dataset, we will need to split it into train/validation/test sets ourselves.    

In [ ]:
base_dir = pathlib.Path("data/handbags-shoes")

In [ ]:
for category in ('handbags', 'shoes'):
    fnames = os.listdir(base_dir / category)

    dir = base_dir / 'train' / category
    os.makedirs(dir, exist_ok=True)
    for fname in fnames[:50]:    # the first 50 examples go into the training set
        shutil.copyfile(src=base_dir / category / fname,
                        dst=dir / fname)

    dir = base_dir / 'validation' / category
    os.makedirs(dir, exist_ok=True)
    for fname in fnames[50:75]:  # the next 25 examples go into the validation set
        shutil.copyfile(src=base_dir / category / fname,
                        dst=dir / fname)

    dir = base_dir / 'test' / category
    os.makedirs(dir, exist_ok=True)
    for fname in fnames[75:]:    # the remaining examples go into the test set
        shutil.copyfile(src=base_dir / category / fname,
                        dst=dir / fname)

The code above creates this directory structure:

train/   
..handbags/         
..shoes/    
validation/    
..handbags/       
..shoes/         
test/    
..handbags/     
..shoes/   




---



---




When working with image JPEGs, we will follow this process:

1.   Read in the JPEGs
2.   Convert the JPEGs into tensors
3.   Resize them to a standard size (since web-scraped images may be in different sizes)
5.   Group them into batches (we'll use batches of 32 images).


In PyTorch, we use `torchvision.datasets.ImageFolder` combined with `torchvision.transforms` to handle all of this.

Note: `transforms.ToTensor()` automatically scales pixel values from [0, 255] to [0, 1] and converts images to (C, H, W) format.

In [ ]:
# Basic transforms: resize, convert to tensor (scales to [0,1])
basic_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

train_dataset = datasets.ImageFolder(base_dir / 'train', transform=basic_transform)
val_dataset = datasets.ImageFolder(base_dir / 'validation', transform=basic_transform)
test_dataset = datasets.ImageFolder(base_dir / 'test', transform=basic_transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print(f"Training examples: {len(train_dataset)}")
print(f"Validation examples: {len(val_dataset)}")
print(f"Test examples: {len(test_dataset)}")
print(f"Classes: {train_dataset.classes}")

**Less than 100 examples in the training set!!**


Let's check the shape of each image. Since these are color images, they will have 3 channels and since we sized them to (224, 224), the shape should be (3, 224, 224) in PyTorch's (C, H, W) format.

In [ ]:
images, labels = next(iter(train_loader))
print(f"Batch image shape: {images.shape}")
print(f"Single image shape: {images[0].shape}")

Check!

Let's look at a few examples.

In [ ]:
plt.figure(figsize=(10, 10))
images, labels = next(iter(train_loader))
for i in range(9):
    ax = plt.subplot(3, 3, i + 1)
    # PyTorch images are (C, H, W), matplotlib expects (H, W, C)
    plt.imshow(images[i].permute(1, 2, 0).numpy())
    plt.title(train_dataset.classes[labels[i]])
    plt.axis("off")
plt.tight_layout()
plt.show()

### Basic Attempt: Vision Transformer from Scratch

In [ ]:
patch_size = 16
image_size = 224

In [ ]:
def get_patches(images, patch_size=16):
    """
    Break images into non-overlapping patches.
    Input:  (B, C, H, W)
    Output: (B, num_patches, patch_size*patch_size*C)
    """
    B, C, H, W = images.shape
    num_patches_h = H // patch_size
    num_patches_w = W // patch_size

    # Reshape to extract patches
    # (B, C, H, W) -> (B, C, num_patches_h, patch_size, num_patches_w, patch_size)
    patches = images.reshape(B, C, num_patches_h, patch_size, num_patches_w, patch_size)
    # -> (B, num_patches_h, num_patches_w, C, patch_size, patch_size)
    patches = patches.permute(0, 2, 4, 1, 3, 5)
    # -> (B, num_patches, patch_dim)
    patches = patches.reshape(B, num_patches_h * num_patches_w, C * patch_size * patch_size)

    return patches

In [ ]:
# Visualize an image and its patches
images, _ = next(iter(train_loader))
sample_image = images[0:1]  # (1, 3, 224, 224)

plt.figure(figsize=(4, 4))
plt.imshow(sample_image[0].permute(1, 2, 0).numpy())
plt.axis("off")
plt.show()

In [ ]:
patches = get_patches(sample_image, patch_size)
no_channels = sample_image.shape[1]
print(f"Image size: {image_size} X {image_size}")
print(f"Patch size: {patch_size} X {patch_size}")
print(f"Patches per image: {patches.shape[1]}")
print(f"Elements per patch: {patches.shape[-1]}")

n = int(np.sqrt(patches.shape[1]))
plt.figure(figsize=(4, 4))
for i, patch in enumerate(patches[0]):
    ax = plt.subplot(n, n, i + 1)
    # Reshape patch back to (C, patch_size, patch_size) then permute for display
    patch_img = patch.reshape(no_channels, patch_size, patch_size).permute(1, 2, 0)
    plt.imshow(patch_img.numpy())
    plt.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# Recall num_patches is akin to the length of our context window
num_patches = (image_size // patch_size) ** 2

# Transformer dimensions
projection_dim = 64
num_heads = 8
transformer_units = [
    projection_dim * 2,
    projection_dim,
]  # no. units in the feedforward portion of the transformers
num_transformer_layers = 2

# Classification head dimensions
mlp_head_units = [
    512,
    256,
]

num_classes = 2
input_channels = 3

In [ ]:
class VisionTransformer(nn.Module):
    """A basic Vision Transformer built from scratch."""

    def __init__(self, image_size, patch_size, projection_dim, num_heads,
                 transformer_units, num_transformer_layers,
                 mlp_head_units, num_classes, num_channels=3):
        super().__init__()
        self.patch_size = patch_size
        self.num_patches = (image_size // patch_size) ** 2
        patch_dim = num_channels * patch_size * patch_size

        # Data augmentation (applied during training only)
        self.augment = nn.Sequential(
            transforms.RandomHorizontalFlip(),
            transforms.RandomRotation(10),
            transforms.RandomResizedCrop(image_size, scale=(0.9, 1.0)),
        )

        # Patch projection
        self.projection = nn.Linear(patch_dim, projection_dim)
        self.position_embedding = nn.Embedding(self.num_patches, projection_dim)

        # Transformer layers
        self.transformer_layers = nn.ModuleList()
        for _ in range(num_transformer_layers):
            self.transformer_layers.append(nn.ModuleDict({
                'norm1': nn.LayerNorm(projection_dim, eps=1e-6),
                'attn': nn.MultiheadAttention(
                    embed_dim=projection_dim,
                    num_heads=num_heads,
                    dropout=0.1,
                    batch_first=True,
                ),
                'norm2': nn.LayerNorm(projection_dim, eps=1e-6),
                'ffn': nn.Sequential(
                    nn.Linear(projection_dim, transformer_units[0]),
                    nn.GELU(),
                    nn.Dropout(0.5),
                    nn.Linear(transformer_units[0], transformer_units[1]),
                    nn.GELU(),
                    nn.Dropout(0.5),
                ),
            }))

        # Classification head
        self.final_norm = nn.LayerNorm(projection_dim, eps=1e-6)
        self.flatten = nn.Flatten()
        self.head_dropout1 = nn.Dropout(0.5)
        head_layers = []
        in_dim = self.num_patches * projection_dim
        for units in mlp_head_units:
            head_layers.extend([
                nn.Linear(in_dim, units),
                nn.GELU(),
                nn.Dropout(0.5),
            ])
            in_dim = units
        head_layers.append(nn.Linear(in_dim, num_classes))
        self.head = nn.Sequential(*head_layers)

    def forward(self, x):
        # Apply data augmentation during training
        if self.training:
            x = self.augment(x)

        # Create patches
        patches = get_patches(x, self.patch_size)  # (B, num_patches, patch_dim)

        # Project patches and add position embeddings
        positions = torch.arange(self.num_patches, device=x.device).unsqueeze(0)
        encoded = self.projection(patches) + self.position_embedding(positions)

        # Transformer blocks
        for layer in self.transformer_layers:
            # Layer norm + multi-head attention + skip connection
            x1 = layer['norm1'](encoded)
            attn_out, _ = layer['attn'](x1, x1, x1)
            x2 = attn_out + encoded
            # Layer norm + FFN + skip connection
            x3 = layer['norm2'](x2)
            x3 = layer['ffn'](x3)
            encoded = x3 + x2

        # Classification head
        x = self.final_norm(encoded)
        x = self.flatten(x)
        x = self.head_dropout1(x)
        logits = self.head(x)
        return logits

In [ ]:
vit_model = VisionTransformer(
    image_size=image_size,
    patch_size=patch_size,
    projection_dim=projection_dim,
    num_heads=num_heads,
    transformer_units=transformer_units,
    num_transformer_layers=num_transformer_layers,
    mlp_head_units=mlp_head_units,
    num_classes=num_classes,
).to(device)

print(vit_model)
num_params = sum(p.numel() for p in vit_model.parameters())
print(f"\nTotal parameters: {num_params:,}")

In [ ]:
def train_model(model, train_loader, val_loader, num_epochs, lr=0.001,
                optimizer_class=torch.optim.AdamW):
    """
    Generic training loop. Returns a history dict with
    loss, accuracy, val_loss, val_accuracy lists.
    """
    criterion = nn.CrossEntropyLoss()
    optimizer = optimizer_class(model.parameters(), lr=lr)
    history = {"loss": [], "accuracy": [], "val_loss": [], "val_accuracy": []}

    for epoch in range(num_epochs):
        # --- Training ---
        model.train()
        epoch_loss, correct, total = 0.0, 0, 0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item() * len(xb)
            correct += (logits.argmax(1) == yb).sum().item()
            total += len(xb)

        train_loss = epoch_loss / total
        train_acc = correct / total

        # --- Validation ---
        model.eval()
        val_loss, val_correct, val_total = 0.0, 0, 0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(device), yb.to(device)
                logits = model(xb)
                loss = criterion(logits, yb)
                val_loss += loss.item() * len(xb)
                val_correct += (logits.argmax(1) == yb).sum().item()
                val_total += len(xb)

        val_loss /= val_total
        val_acc = val_correct / val_total

        history["loss"].append(train_loss)
        history["accuracy"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_accuracy"].append(val_acc)

        print(f"Epoch {epoch+1}/{num_epochs} - "
              f"loss: {train_loss:.4f} - accuracy: {train_acc:.4f} - "
              f"val_loss: {val_loss:.4f} - val_accuracy: {val_acc:.4f}")

    return history

In [ ]:
def evaluate_model(model, test_loader):
    """Evaluate a model on a test DataLoader. Returns (loss, accuracy)."""
    criterion = nn.CrossEntropyLoss()
    model.eval()
    test_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for xb, yb in test_loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            loss = criterion(logits, yb)
            test_loss += loss.item() * len(xb)
            correct += (logits.argmax(1) == yb).sum().item()
            total += len(xb)
    test_loss /= total
    test_acc = correct / total
    print(f"Test loss: {test_loss:.4f}")
    print(f"Test accuracy: {test_acc:.4f}")
    return test_loss, test_acc

In [ ]:
def plot_loss_curves(history):
    plt.clf()
    epochs = range(1, len(history["loss"]) + 1)
    plt.plot(epochs, history["loss"], "bo", label="Training loss")
    plt.plot(epochs, history["val_loss"], "b", label="Validation loss")
    plt.title("Training and validation loss")
    plt.xlabel("Epochs")
    plt.ylabel("Loss")
    plt.legend()
    plt.show()

def plot_acc_curves(history):
    plt.clf()
    epochs = range(1, len(history["accuracy"]) + 1)
    plt.plot(epochs, history["accuracy"], "bo", label="Training acc")
    plt.plot(epochs, history["val_accuracy"], "b", label="Validation acc")
    plt.title("Training and validation accuracy")
    plt.xlabel("Epochs")
    plt.ylabel("Accuracy")
    plt.legend()
    plt.show()

In [ ]:
history = train_model(vit_model, train_loader, val_loader, num_epochs=40, lr=0.001)

### Finetuning a Pretrained ViT

In [ ]:
# Transforms with ImageNet normalization for pretrained ViT
imagenet_normalize = transforms.Normalize(
    mean=[0.485, 0.456, 0.406],
    std=[0.229, 0.224, 0.225],
)

vit_train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.RandomResizedCrop(224, scale=(0.9, 1.0)),
    transforms.ToTensor(),
    imagenet_normalize,
])

vit_eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    imagenet_normalize,
])

vit_train_dataset = datasets.ImageFolder(base_dir / 'train', transform=vit_train_transform)
vit_val_dataset = datasets.ImageFolder(base_dir / 'validation', transform=vit_eval_transform)
vit_test_dataset = datasets.ImageFolder(base_dir / 'test', transform=vit_eval_transform)

vit_train_loader = DataLoader(vit_train_dataset, batch_size=32, shuffle=True)
vit_val_loader = DataLoader(vit_val_dataset, batch_size=32, shuffle=False)
vit_test_loader = DataLoader(vit_test_dataset, batch_size=32, shuffle=False)

In [ ]:
# Load pretrained ViT-B/16
vit_backbone = vit_b_16(weights=ViT_B_16_Weights.IMAGENET1K_V1)

# Freeze the backbone
for param in vit_backbone.parameters():
    param.requires_grad = False

# Replace the classification head
# The ViT head is at model.heads.head
in_features = vit_backbone.heads.head.in_features
vit_backbone.heads.head = nn.Sequential(
    nn.Linear(in_features, 256),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(256, 2),
)

vit_backbone = vit_backbone.to(device)
print(f"ViT trainable parameters: {sum(p.numel() for p in vit_backbone.parameters() if p.requires_grad):,}")

In [ ]:
# Train only the classification head
history = train_model(vit_backbone, vit_train_loader, vit_val_loader,
                      num_epochs=2, lr=1e-4, optimizer_class=torch.optim.Adam)

In [ ]:
evaluate_model(vit_backbone, vit_test_loader)

## A Basic Convolutional Neural Network


---



---



We will try a simple CNN on this dataset with two convolutional blocks.

In [ ]:
class BasicCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            # First convolutional block
            nn.Conv2d(3, 32, kernel_size=2),   # Conv_1
            nn.ReLU(),
            nn.MaxPool2d(2),

            # Second convolutional block
            nn.Conv2d(32, 32, kernel_size=2),  # Conv_2
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        # Calculate the flattened size: input (3, 224, 224)
        # After Conv_1(k=2): (32, 223, 223), MaxPool: (32, 111, 111)
        # After Conv_2(k=2): (32, 110, 110), MaxPool: (32, 55, 55)
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 55 * 55, 2),
        )

    def forward(self, x):
        # Note: ToTensor() already scales to [0,1],
        # so no Rescaling(1./255) layer is needed.
        x = self.features(x)
        x = self.classifier(x)
        return x

cnn_model = BasicCNN().to(device)
print(cnn_model)
num_params = sum(p.numel() for p in cnn_model.parameters())
print(f"\nTotal parameters: {num_params:,}")

We use `CrossEntropyLoss` which combines log-softmax and negative log-likelihood. With 2 output units this is equivalent to binary cross-entropy.

We will store the results of training in the variable `history`. This will allow us to plot how the loss and accuracy changed from epoch to epoch and thereby get a sense for any overfitting.

In [ ]:
history = train_model(cnn_model, train_loader, val_loader, num_epochs=20, lr=0.001,
                      optimizer_class=torch.optim.Adam)

In [ ]:
plot_loss_curves(history)

In [ ]:
plot_acc_curves(history)

The model achieves a high training accuracy. This is not surprising, since our training dataset has only ~90 examples while our model has many parameters!

The validation accuracy curve is very noisy because the validation dataset is only 49 examples but it is clear there's a gap between training and validation accuracy curves, suggesting **overfitting**.




Let's check the accuracy on the test set.

In [ ]:
evaluate_model(cnn_model, test_loader)

OK, what can we do to improve accuracy?

We can go back and scrape more data, of course, but that is a lot of work.

What else?

## Data Augmentation for Images

The basic idea of augmentation is to slightly alter the image so that the value of the dependent variable (i.e. the category that it belongs to) doesn't change. For instance, if you rotate the image of a handbag by 10 degrees or zoom in on it slightly, the content of the image doesn't change; it is *still* a handbag.

By applying these transformations repeatedly to an image, you can create new images and thereby increase the size of the dataset almost effortlessly.

Researchers have developed a list of these transformations that you can apply to images and PyTorch provides them out of the box via `torchvision.transforms`.



---



---



To demonstrate, here's a little function that applies three transformations to an incoming image.


In [ ]:
augment_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
])

Let's apply it to an image from our training set to see what it produces.

In [ ]:
plt.figure(figsize=(10, 10))
images, _ = next(iter(train_loader))
sample = images[0]  # (C, H, W) tensor
for i in range(9):
    ax = plt.subplot(3, 3, i + 1)
    augmented = augment_transform(sample)
    plt.imshow(augmented.permute(1, 2, 0).numpy())
    plt.axis("off")
plt.tight_layout()
plt.show()

We can apply data augmentation via the transforms pipeline in our DataLoader. In PyTorch, data augmentation is applied in the data loading pipeline (not as model layers), and it naturally only applies during training since we use different transforms for train vs. validation/test.

In [ ]:
# Training transforms with augmentation
augmented_train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.ToTensor(),
])

aug_train_dataset = datasets.ImageFolder(base_dir / 'train', transform=augmented_train_transform)
aug_train_loader = DataLoader(aug_train_dataset, batch_size=32, shuffle=True)

In [ ]:
class AugmentedCNN(nn.Module):
    """Same CNN architecture as BasicCNN — augmentation is in the DataLoader."""
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            # First convolutional block
            nn.Conv2d(3, 32, kernel_size=2),
            nn.ReLU(),
            nn.MaxPool2d(2),

            # Second convolutional block
            nn.Conv2d(32, 32, kernel_size=2),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 55 * 55, 2),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

aug_cnn_model = AugmentedCNN().to(device)
print(aug_cnn_model)

In [ ]:
history = train_model(aug_cnn_model, aug_train_loader, val_loader,
                      num_epochs=10, lr=0.001, optimizer_class=torch.optim.Adam)

In the interest of time, we won't fully train this model. We encourage you to do so and evaluate if the data augmentation improves the accuracy on the test set.

## Model built with Transfer Learning


**SWITCH TO PPT FOR TRANSFER LEARNING**



---

We will work with ResNet-50 since it comes pre-packaged with PyTorch's `torchvision`.

Check out all the pre-trained models available in [torchvision](https://pytorch.org/vision/stable/models.html).



In [ ]:
# Load ResNet-50 pretrained on ImageNet, without the final classification layer
resnet50_backbone = resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)

# Remove the final fully-connected layer (the "head")
# We'll extract features from everything before fc
# Freeze all backbone parameters
for param in resnet50_backbone.parameters():
    param.requires_grad = False

# Replace the final fc layer with identity to get features
resnet50_backbone.fc = nn.Identity()
resnet50_backbone = resnet50_backbone.to(device)
resnet50_backbone.eval()

print("ResNet-50 backbone loaded (frozen, headless).")

It is a **DEEP** network, all right!!

Next, we run our dataset through "headless ResNet" to get the transformed, "smart" inputs.


In [ ]:
# Create datasets with ImageNet normalization for ResNet
resnet_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    imagenet_normalize,
])

resnet_train_dataset = datasets.ImageFolder(base_dir / 'train', transform=resnet_transform)
resnet_val_dataset = datasets.ImageFolder(base_dir / 'validation', transform=resnet_transform)
resnet_test_dataset = datasets.ImageFolder(base_dir / 'test', transform=resnet_transform)

resnet_train_loader = DataLoader(resnet_train_dataset, batch_size=32, shuffle=False)
resnet_val_loader = DataLoader(resnet_val_dataset, batch_size=32, shuffle=False)
resnet_test_loader = DataLoader(resnet_test_dataset, batch_size=32, shuffle=False)

In [ ]:
def get_features_and_labels(backbone, loader):
    """Extract features from a frozen backbone for all data in a DataLoader."""
    all_features = []
    all_labels = []
    backbone.eval()
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            features = backbone(images)
            all_features.append(features.cpu())
            all_labels.append(labels)
    return torch.cat(all_features), torch.cat(all_labels)

In [ ]:
train_features, train_labels = get_features_and_labels(resnet50_backbone, resnet_train_loader)
val_features, val_labels = get_features_and_labels(resnet50_backbone, resnet_val_loader)
test_features, test_labels = get_features_and_labels(resnet50_backbone, resnet_test_loader)

What's the shape of the tensor that comes out of 'headless' ResNet?

In [ ]:
print(f"Feature shape: {train_features.shape}")

These tensors coming out of "headless" ResNet are smart representations and we can simply attach them to a small NN.

We will use a regularization layer that we haven't yet used: `Dropout`.

In [ ]:
class TransferHead(nn.Module):
    """Small classification head on top of ResNet-50 features."""
    def __init__(self, in_features):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_features, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, 2),
        )

    def forward(self, x):
        return self.net(x)

in_features = train_features.shape[1]
head_model = TransferHead(in_features).to(device)
print(head_model)
print(f"\nTotal parameters: {sum(p.numel() for p in head_model.parameters()):,}")

In [ ]:
# Create DataLoaders from the precomputed features
from torch.utils.data import TensorDataset

feature_train_loader = DataLoader(
    TensorDataset(train_features, train_labels), batch_size=32, shuffle=True)
feature_val_loader = DataLoader(
    TensorDataset(val_features, val_labels), batch_size=32, shuffle=False)
feature_test_loader = DataLoader(
    TensorDataset(test_features, test_labels), batch_size=32, shuffle=False)

In [ ]:
history = train_model(head_model, feature_train_loader, feature_val_loader,
                      num_epochs=10, lr=0.001, optimizer_class=torch.optim.Adam)

In [ ]:
plot_acc_curves(history)

The training and validation accuracies are both very high! This looks promising!

In [ ]:
evaluate_model(head_model, feature_test_loader)

**IMPRESSIVE ACCURACY ON THE TEST SET!!**


Let's pause for a moment to reflect on what we have done.

We have built an amazingly accurate handbags or shoes classifier with *just* 100 training images!

That's the power of transfer learning!!

OK, let's test it with a prediction function.

In [ ]:
def predict_image(image_path, backbone, head, class_names):
    """Load an image, extract ResNet features, and predict its class."""
    img = Image.open(image_path).convert('RGB')
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        imagenet_normalize,
    ])
    img_tensor = transform(img).unsqueeze(0).to(device)

    backbone.eval()
    head.eval()
    with torch.no_grad():
        features = backbone(img_tensor)
        logits = head(features)
        probs = torch.softmax(logits, dim=1)
        pred_idx = probs.argmax(1).item()
        confidence = probs[0, pred_idx].item()

    pred_label = class_names[pred_idx]
    print("************************************\n")
    print(f"...........it is a {pred_label.upper()}! ({confidence:.1%} confidence)")
    print("\n************************************\n")

    plt.imshow(img)
    plt.title(f"Prediction: {pred_label} ({confidence:.1%})")
    plt.axis('off')
    plt.show()

    return pred_label, confidence

In [ ]:
# Test with an image from the test set
test_images = list((base_dir / 'test' / 'handbags').iterdir())
if test_images:
    predict_image(test_images[0], resnet50_backbone, head_model,
                  class_names=resnet_train_dataset.classes)

**THE END**